# 01_fastmcp_tools

Stage 1 of the `codes/mcp/` track: MCP tools loop on **FastMCP + Pydantic**.

Notebook path: `codes/mcp/01_fastmcp_tools/qa.ipynb`. File I/O for this stage
stays in this directory (`server.py`, `client.py`). Shared deps live in
`codes/requirements.txt` (one file for the whole `codes/` lab tree).

Install once (venv recommended), from this stage directory:

```bash
pip install -r ../../requirements.txt
```

Run the next cell first to bind `ROOT`. Edit only under `# answer`. Pre-filled
setup code stays as-is. Get each cell running before moving on.

Architecture to keep in your head:

`Client(mcp) → (framework negotiates 2026-07-28) → list_tools / call_tool → @mcp.tool`

You do **not** hand-write JSON-RPC envelopes or `inputSchema` dicts. Type hints
+ `pydantic.Field` become the schema; FastMCP owns the protocol.

FastMCP's Client is async — cells below use top-level `await` (Jupyter /
IPython). From a plain script, wrap the same coroutine in `asyncio.run(...)`.

In [1]:
from pathlib import Path

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "01_fastmcp_tools" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "mcp" / "01_fastmcp_tools"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/mcp/01_fastmcp_tools")

ROOT = lab_root()
print(ROOT)

/workspace/AEFS-Notes/codes/mcp/01_fastmcp_tools


## 1. Read a framework-generated `tools/list` shape

`payload` is what a host sees after `list_tools` — names plus a JSON Schema that
**FastMCP derived from Pydantic / annotations**, not a dict you typed by hand.

Print every tool `name`, then print the `required` fields of the tool named `add`.

In [2]:
payload = {
    "tools": [
        {
            "name": "echo",
            "description": "Return the given text unchanged.",
            "inputSchema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "text": {
                        "description": "Text to return unchanged",
                        "type": "string",
                    }
                },
                "required": ["text"],
            },
        },
        {
            "name": "add",
            "description": "Add two numbers and return the sum as text.",
            "inputSchema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "a": {"description": "Left operand", "type": "number"},
                    "b": {"description": "Right operand", "type": "number"},
                },
                "required": ["a", "b"],
            },
        },
    ]
}

# answer

names = [tool["name"] for tool in payload["tools"]]
print("names:", names)
add = next(tool for tool in payload["tools"] if tool["name"] == "add")
print("add.required:", add["inputSchema"]["required"])

names: ['echo', 'add']
add.required: ['a', 'b']


## 2. Let FastMCP + Pydantic build the schema

Create a tiny `FastMCP` server in this cell with one tool `greet(name: str)`
annotated with `Field(description=...)`. Then use `Client(server)` to
`list_tools()` and print:

1. the negotiated `protocol_version` (expect `2026-07-28`)
2. the tool `name`
3. `input_schema["required"]`

No hand-rolled JSON Schema dicts.

In [3]:
from typing import Annotated

from fastmcp import Client, FastMCP
from pydantic import Field

# answer

server = FastMCP("qa-greet")

@server.tool
def greet(
    name: Annotated[str, Field(description="Who to greet")],
) -> str:
    """Return a short greeting."""
    return f"hello {name}"

async with Client(server) as client:
    print("protocol:", client.protocol_version)
    tools = await client.list_tools()
    tool = tools[0]
    print("name:", tool.name)
    print("required:", tool.input_schema["required"])

protocol: 2026-07-28
name: greet
required: ['name']


## 3. Drive the lab server: list → call

Import `mcp` from `server.py` in this directory. With `Client(mcp)`:

1. print `protocol_version` and `server_info.name`
2. `list_tools` and print tool names
3. `call_tool("echo", {"text": "ping"})` and print the text content

This is the same initialize → tools/list → tools/call loop — FastMCP performs
the handshake when the client context enters.

In [4]:
import sys

sys.path.insert(0, str(ROOT))
from fastmcp import Client
from server import mcp

# answer

async with Client(mcp) as client:
    print("protocol:", client.protocol_version)
    print("server:", client.server_info.name if client.server_info else None)
    tools = await client.list_tools()
    print("tools:", [t.name for t in tools])
    result = await client.call_tool("echo", {"text": "ping"})
    print("echo:", result.content[0].text)

protocol: 2026-07-28
server: lab-tools-server
tools: ['echo', 'add']


echo: ping


## 4. Register a third tool

Build a local `FastMCP("qa-ext")` that also exposes `mul(a, b)` with
`Annotated` + `Field`, then `call_tool("mul", {"a": 4, "b": 5})` and print the
returned text (`"20.0"` or `"20"` are both fine).

Extension point for later stages: keep adding ordinary typed callables;
transport / protocol stay with FastMCP.

In [5]:
# answer

ext = FastMCP("qa-ext")

@ext.tool
def mul(
    a: Annotated[float, Field(description="Left operand")],
    b: Annotated[float, Field(description="Right operand")],
) -> str:
    """Multiply two numbers and return the product as text."""
    return str(a * b)

async with Client(ext) as client:
    result = await client.call_tool("mul", {"a": 4, "b": 5})
    print(result.content[0].text)

20.0
